# Traffic Flow Analyzer (RTD cell tracking) — Google Colab

Runs `traffic_analyzer.py` (the Python 3 / headless-compatible port of
[Matthew Thomas's Traffic Flow Analyzer](https://github.com/telescope7/TrafficFlowAnalysis),
used per the RTD paper's method — see `PORTING_NOTES.md` in this repo) against
an uploaded video, with the paper's parameters as defaults, and plots the
resulting tracks.

`cv2.imshow`/interactive trackbars don't work in Colab (no display), so this
notebook runs the tool in **batch mode** and writes an annotated output video
with `-o` you can review afterward instead.

## 1. Get `traffic_analyzer.py`

If you opened this notebook directly from the GitHub repo, the script is
already alongside it — this cell clones the repo as a fallback (e.g. if
you uploaded just this `.ipynb` on its own).

In [ ]:
import os

REPO_URL = "https://github.com/goeulla/cells"  # update if your fork lives elsewhere

if not os.path.exists("traffic_analyzer.py"):
    !git clone --depth 1 {REPO_URL} _cells_repo
    !cp _cells_repo/traffic_analyzer.py .
    !cp _cells_repo/requirements-traffic_analyzer.txt . 2>/dev/null || true

assert os.path.exists("traffic_analyzer.py"), "traffic_analyzer.py not found — upload it manually via the Files pane."

## 2. Install dependencies

`opencv-python-headless` (not plain `opencv-python`) — Colab has no display,
and the headless wheel is the one that installs cleanly here.

In [ ]:
!pip install -q opencv-python-headless numpy pandas matplotlib

## 3. Provide your video

Either upload a file, or mount Google Drive and point `VIDEO_PATH` at a file
already there (better for large videos than uploading through the browser).

In [ ]:
# Option A: upload directly (good for smaller videos)
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded)) if uploaded else None
print("Using video:", VIDEO_PATH)

In [ ]:
# Option B: mount Drive instead (skip Option A above, set VIDEO_PATH yourself)
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = "/content/drive/MyDrive/path/to/video.avi"

## 4. Set analysis parameters

Defaults below match the RTD paper's method (background subtraction, mask
weight 0.0005, blur 13, threshold 5). `OBJECT_DIAMETER_MIN_PX` is the one
parameter you must calibrate yourself: the paper's cutoff is 5 um, but this
tool works in pixels, so convert using your video's um-per-pixel scale
(`UM_PER_PIXEL`, also used below to convert output back to physical units).

In [ ]:
UM_PER_PIXEL = 1.0  # <-- set this to your video's calibration (um per pixel)

BLUR = 13
THRESHOLD = 5
ACCUMULATOR_WEIGHT = 0.0005
OBJECT_DIAMETER_MIN_UM = 5.0
OBJECT_DIAMETER_MIN_PX = max(1, round(OBJECT_DIAMETER_MIN_UM / UM_PER_PIXEL))

OUTPUT_CSV = "tracks.csv"
OUTPUT_VIDEO = "tracks_annotated.avi"

print("object diameter min:", OBJECT_DIAMETER_MIN_PX, "px")

## 5. Run the analysis (batch mode)

In [ ]:
assert VIDEO_PATH, "Set VIDEO_PATH first (upload a video in step 3, or mount Drive and set it manually)."

!python3 traffic_analyzer.py \
  -m "{VIDEO_PATH}" \
  -c {OBJECT_DIAMETER_MIN_PX} \
  -b {BLUR} \
  -t {THRESHOLD} \
  -s -a {ACCUMULATOR_WEIGHT} \
  -o "{OUTPUT_VIDEO}" \
  > "{OUTPUT_CSV}"

!wc -l "{OUTPUT_CSV}"

### Optional: quick inline visual sanity check

`-v` has no live interactive window in Colab, but it will preview every Nth
annotated frame inline (via `google.colab.patches.cv2_imshow`) so you can
eyeball whether blur/threshold/diameter look reasonable before trusting the
full run above. This reruns the analysis, so skip it for large videos.

In [ ]:
# !python3 traffic_analyzer.py -m "{VIDEO_PATH}" -c {OBJECT_DIAMETER_MIN_PX} \
#   -b {BLUR} -t {THRESHOLD} -s -a {ACCUMULATOR_WEIGHT} \
#   -v --colab_preview_stride 30 > /dev/null

## 6. Load results and apply the paper's physical-unit filters

The paper keeps tracks with tracking distance > 400 px, cell diameter
between 14-60 um, and within 50 um of the chamber bottom (by tracked
velocity). The chamber-bottom check depends on your experimental setup and
isn't computed here — apply it to `df` however fits your data.

In [ ]:
import pandas as pd
import numpy as np

COLUMNS = [
    "num_frames", "first_x", "first_y", "last_x", "last_y",
    "first_frame", "last_frame", "avg_radius", "avg_width",
    "avg_height", "avg_area",
]

df = pd.read_csv(OUTPUT_CSV, header=None, names=COLUMNS)

df["tracking_distance_px"] = np.hypot(df.last_x - df.first_x, df.last_y - df.first_y)
df["avg_diameter_um"] = df.avg_radius * 2 * UM_PER_PIXEL

paper_filtered = df[
    (df.tracking_distance_px > 400)
    & (df.avg_diameter_um.between(14, 60))
    # & <your within-50um-of-chamber-bottom condition here>
]

print(f"{len(df)} tracked objects total, {len(paper_filtered)} pass the paper's distance/diameter filters")
df.head()

## 7. Plot velocities

Requires the video's frame rate (fps) to convert pixel displacement/frame
into a real speed.

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
cap.release()
print("video fps:", fps)

elapsed_s = (paper_filtered.last_frame - paper_filtered.first_frame) / fps
speed_um_s = (paper_filtered.tracking_distance_px * UM_PER_PIXEL) / elapsed_s

plt.hist(speed_um_s, bins=30)
plt.xlabel("speed (um/s)")
plt.ylabel("count")
plt.title("Cell velocity distribution")
plt.show()

## 8. Download results

In [ ]:
from google.colab import files

files.download(OUTPUT_CSV)
if os.path.exists(OUTPUT_VIDEO):
    files.download(OUTPUT_VIDEO)